In [53]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime

In [54]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions, SamplerOptions
from qiskit_aer import AerSimulator
import numpy as np
from numpy import pi
from matplotlib import pyplot as plt
import matplotlib
from scipy.optimize import minimize
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit import Parameter
from scipy.optimize import minimize
from scipy.linalg import eigh
from scipy.special import erf
from functools import partial
from qiskit_aer import AerSimulator
from qiskit.circuit.classical import expr

## Measurement


In [55]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [56]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [57]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [58]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

## Single qubit clifford group

In [59]:
def add_S(qc, q, cbit):
    c = cbit

    # (ancilla, data) = (q+1, q) initilize ancilla qubit in |+>
    qc.h(q+1)

    measure_XI(qc, q+1, q, c[0])
    measure_ZZ(qc, q+1, q, c[1])
    measure_YI(qc, q+1, q, c[2])
    measure_XI(qc, q+1, q, c[3])

    # measurement bit c_i encodes s_i = (-1)^{c_i}
    # s_i s_j = (-1)^{c[i] + c[j]}
    # product becomes XOR

    # Z^{(1 + s0 s1 s2)/2}
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(q)

    qc.reset(q+1) # reset ancilla to |0> to make test easy

    return qc

In [60]:
def add_HSH(qc, q, cbit):
    c = cbit

    qc.h(q+1)

    measure_XI(qc, q+1, q, c[0])
    measure_ZZ(qc, q+1, q, c[1])
    measure_ZY(qc, q+1, q, c[2])
    measure_XI(qc, q+1, q, c[3])

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(q)

    # Y^{(1 - s0 s3)/2} odd parity
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(q)

    qc.reset(q+1)

    return qc

In [61]:
def add_HS(qc, q, cbit):
    c = cbit

    qc.h(q+1)

    measure_XI(qc, q+1, q, c[0])  # s0
    measure_ZY(qc, q+1, q, c[1])  # s1
    measure_ZZ(qc, q+1, q, c[2])  # s2
    measure_YI(qc, q+1, q, c[3])  # s3
    measure_XI(qc, q+1, q, c[4])  # s4

    # Z^{(1 + s0 s1 s3)/2} even parity
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(q)

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(q)

    qc.reset(q+1)

    return qc

## Single qubit clifford gate in supremacy circuit

In [62]:
def add_sqrtX(qc, q, cbit):
    add_HSH(qc, q, cbit)
    return qc

In [63]:
def add_sqrtY(qc, q, cbit):
    add_S(qc, q, cbit)
    add_HS(qc, q, cbit)
    return qc

## Test on $|0\rangle$ and $|+\rangle$ to cover the whole Hilbert space of qubit

All circuits with single-qubit clifford gates need at most 5 classical bits

In [64]:
# sqrt(X) test — input |0>
# remember sqrt(X) rotates Z to -Y, final state is |-y>

# 1) direct sqrt(X)
qc1 = QuantumCircuit(2)
qc1.sx(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(X) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)  # need at most 5 classical bits
qc2.add_register(cbit2)
add_sqrtX(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(X) on |0>:")
print(result.data(0)["statevector"])

direct sqrt(X) on |0>:
[0.5+0.5j 0.5-0.5j 0. +0.j  0. +0.j ]
MBQC sqrt(X) on |0>:
{'0xc': Statevector([-7.07106781e-01+2.16489014e-16j,
              2.16489014e-16+7.07106781e-01j,
              0.00000000e+00+0.00000000e+00j,
              0.00000000e+00-0.00000000e+00j],
            dims=(2, 2))}


In [65]:
# sqrt(X) test — input |+>

# 1) direct sqrt(X)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.sx(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(X) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)
qc2.add_register(cbit2)
qc2.h(0)
add_sqrtX(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(X) on |+>:")
print(result.data(0)["statevector"])

direct sqrt(X) on |+>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]
MBQC sqrt(X) on |+>:
{'0x2': Statevector([0.5-0.5j, 0.5-0.5j, 0. +0.j , 0. +0.j ],
            dims=(2, 2))}


In [66]:
# sqrt(Y) test — input |0>
# remember sqrt(Y) rotates Z to X, final state is |+>

# 1) direct sqrt(Y)
qc1 = QuantumCircuit(2)
qc1.ry(np.pi/2, 0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(Y) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(Y)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)
qc2.add_register(cbit2)
add_sqrtY(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(Y) on |0>:")
print(result.data(0)["statevector"])

direct sqrt(Y) on |0>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]
MBQC sqrt(Y) on |0>:
{'0xe': Statevector([3.92523115e-16+0.70710678j, 4.71027738e-16+0.70710678j,
             0.00000000e+00+0.j        , 0.00000000e+00+0.j        ],
            dims=(2, 2))}


In [67]:
# sqrt(Y) test — input |+>

# 1) direct sqrt(Y)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.ry(np.pi/2, 0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(Y) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(Y)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)
qc2.add_register(cbit2)
qc2.h(0)
add_sqrtY(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(Y) on |+>:")
print(result.data(0)["statevector"])

direct sqrt(Y) on |+>:
[8.86511593e-17+0.j 1.00000000e+00+0.j 0.00000000e+00+0.j
 0.00000000e+00+0.j]
MBQC sqrt(Y) on |+>:
{'0x1e': Statevector([-1.38777878e-16-2.77555756e-17j,
              1.00000000e+00-1.06452016e-15j,
              0.00000000e+00+0.00000000e+00j,
             -0.00000000e+00+0.00000000e+00j],
            dims=(2, 2))}
